### IMPORT LIBRARY

In [26]:
import re
import unicodedata
import pdfplumber
from collections import defaultdict
from cleantext import clean
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

### LOAD MODEL

In [ ]:
MODEL_PATH = "landing_page/backend/final_bert_model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

print(f"✓ Model loaded from {MODEL_PATH}")
print("Labels:", model.config.id2label)

coba = ner_pipeline("Saya lulusan S1 Teknik Informatika dari Universitas Indonesia dan memiliki keahlian dalam Python, Machine Learning, dan Data Analysis.")
print(coba)

Device set to use cpu


✓ Model loaded from landing_page/backend/final_best_model
Labels: {0: 'O', 1: 'B-EDUCATION', 2: 'I-EDUCATION', 3: 'B-SKILL', 4: 'I-SKILL'}
[{'entity_group': 'EDUCATION', 'score': np.float32(1.0), 'word': 'Saya lulusan S1 Teknik Informatika dari Universitas Indonesia dan memiliki keahlian dalam Python, Machine Learning,', 'start': 0, 'end': 115}, {'entity_group': 'EDUCATION', 'score': np.float32(0.99999976), 'word': '.', 'start': 133, 'end': 134}]


### READ AND CLEAN CV

In [17]:
pdf_path = "pdf_cvs/cv-ardi.pdf"

with pdfplumber.open(pdf_path) as pdf:
    text_before = ""
    for i, page in enumerate(pdf.pages, start=1):
        page_text = page.extract_text() or ""
        text_before += f"\n--- Halaman {i} ---\n{page_text}\n"

print(f"Data CV sebelum dibersihkan:\n{text_before}")

Data CV sebelum dibersihkan:

--- Halaman 1 ---
.
Ardiansyah Indra Febrianto
Surabaya, East Java, Indonesia indrardi92@gmail.com 087761135290 in/ardiansyahfebrianto
SUMMARY
A passionate third-year Data Science student with strong analytical skills and a keen interest in solving real-world problems. Experienced as a data analyst with a focus on
statistics and data flow. Enthusiastic about continuous learning and skill development, actively seeking opportunities to apply data-driven solutions in practical scenarios.
EDUCATION
Applied Data Science
Politeknik Elektronika Negeri Surabaya • Surabaya, Indonesia• 2026 • 3.67 / 4.0
Natural Science Major
Sekolah Menengah Atas 4 Surabaya• Surabaya• 2022
EXPERIENCE
Digital Media Creative Team Member
Applied Data Science May 2023 - Present, Surabaya, Indonesia
• Developed and executed creative content strategies to support the branding of the Data Science program.
• Designed engaging Instagram feed posts, including fun facts about Data Science, adm

In [18]:
import pdfplumber, re, unicodedata
from cleantext import clean

with pdfplumber.open(pdf_path) as pdf:
    text = " ".join(page.extract_text() or "" for page in pdf.pages)

text = unicodedata.normalize("NFKD", text)
text = re.sub(r"[●•▪◆■□▲▼▶◀※☆★→←⇔|]", " ", text)
text = re.sub(r"\[.*?\]", " ", text)
# text = re.sub(r"\s+", " ", text).strip()
text = clean(text,
             clean_all=False,
             extra_spaces=True,
             lowercase=False,
             numbers=True,
             punct=True,
             reg_replace=" ",
             stp_lang="english")

print(f"Data CV sesudah dibersihkan:\n{text}")

Data CV sesudah dibersihkan:
Ardiansyah Indra Febrianto
Surabaya East Java Indonesia indrardigmailcom  inardiansyahfebrianto
SUMMARY
A passionate thirdyear Data Science student with strong analytical skills and a keen interest in solving realworld problems Experienced as a data analyst with a focus on
statistics and data flow Enthusiastic about continuous learning and skill development actively seeking opportunities to apply datadriven solutions in practical scenarios
EDUCATION
Applied Data Science
Politeknik Elektronika Negeri Surabaya Surabaya Indonesia    
Natural Science Major
Sekolah Menengah Atas  Surabaya Surabaya 
EXPERIENCE
Digital Media Creative Team Member
Applied Data Science May   Present Surabaya Indonesia
 Developed and executed creative content strategies to support the branding of the Data Science program
 Designed engaging Instagram feed posts including fun facts about Data Science admission pathways and congratulatory messages for outstanding students
 Utilized graph

### SEGMENTASI BY HEADER

In [19]:
HEADERS = {
    "education": ["education", "academic background"],
    "skills": ["technical skills", "skills", "tools & technologies"],
    # tambahkan header lain untuk deteksi stop
    "experience": ["experience", "work experience","EXPERIENCE"],
    "projects": ["projects", "project"],
    "organization": ["organization", "organizations"],
    "certifications": ["certifications", "certification"],
}

def segment_text(text):
    segments = {key: [] for key in HEADERS}
    current_section = None

    for line in text.split():
        line_lower = line.strip().lower()
        matched_section = None
        for key, aliases in HEADERS.items():
            if any(alias in line_lower for alias in aliases):
                matched_section = key
                break

        if matched_section:
            # kalau ketemu header baru, pindah section
            current_section = matched_section
            continue

        if current_section:
            # kalau sedang di education/skills, tapi ketemu kata yang mirip header lain → stop
            if current_section in ["education", "skills"]:
                for other_key, aliases in HEADERS.items():
                    if other_key not in ["education", "skills"]:
                        if any(alias in line_lower for alias in aliases):
                            current_section = None  # stop menulis ke education/skills
                            break

            if current_section:
                segments[current_section].append(line)

    return {k: " ".join(v).strip() for k, v in segments.items() if v}

# contoh pemakaian
segments = segment_text(text)
print("Education:", segments.get("education", ""))
print("Skills:", segments.get("skills", ""))


Education: Applied Data Science Politeknik Elektronika Negeri Surabaya Surabaya Indonesia Natural Science Major Sekolah Menengah Atas Surabaya Surabaya Development of a MultiNode Greenhouse Condition Dashboard Based on Data Mining as an Effort to Empower Melon Farmers in Wates District Blitar Regency Republik Melon dashtaniid December Present Backend Developer Flask Supabase cPanel Developed backend services using Flask framework to handle API requests and manage realtime data processing Integrated Supabase for database management ensuring seamless connection and data retrieval Deployed machine learning models and backend applications on cPanel hosting handling server configurations and troubleshooting Dashboard Developer Reactjs Realtime Monitoring Designed and developed a responsive dashboard interface using Reactjs for realtime data visualization Implemented dynamic monitoring solutions displaying key metrics such as soil pH room temperature humidity and light intensity from greenho

### NER

In [20]:
def merge_entities(results):
    """
    Gabungkan subword tokens (##) menjadi entitas utuh.
    """
    print(f"Ini adalah hasil results: {results}")
    merged = []
    current_label, current_word = None, []
    for e in results:
        label = e.get("entity_group", "").upper()
        word = (e.get("word", "") or "").replace("##", "")
        if not label or not word:
            continue
        if label == current_label:
            current_word.append(word)
        else:
            if current_word:
                merged.append({"entity_group": current_label, "word": "".join(current_word)})
            current_label, current_word = label, [word]
    if current_word:
        merged.append({"entity_group": current_label, "word": "".join(current_word)})
    return merged


def extract_entities(text: str, model_pipeline, chunk_size: int = 3000, overlap: int = 200):
    """
    Ekstrak entitas dari teks (education & skills) dengan merge subword tokens.
    """
    if not text.strip():
        return {"education": [], "skills": [], "raw": []}

    entities = {"education": set(), "skills": set()}
    raw_entities = []

    # Chunking untuk teks panjang
    step = max(1, chunk_size - overlap)
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), step)]

    for chunk in chunks:
        merged = merge_entities(model_pipeline(chunk))
        for e in merged:
            label = e["entity_group"]
            word = e["word"].strip()
            if not word:
                continue
            raw_entities.append((label, word))
            if "EDUCATION" in label:
                entities["education"].add(word)
            elif "SKILL" in label:
                entities["skills"].add(word)

    return {
        "education": list(entities["education"]),
        "skills": list(entities["skills"]),
        "raw": raw_entities
    }


# ---- Pemakaian yang benar: jalankan sekali per segmen ----
edu_entities = extract_entities(segments.get("education", ""), ner_pipeline)
skill_entities = extract_entities(segments.get("skills", ""), ner_pipeline)

cv_entities = {
    "education": edu_entities["education"],
    "skills": skill_entities["skills"],
    "raw": edu_entities["raw"] + skill_entities["raw"]
}


Ini adalah hasil results: [{'entity_group': 'EDUCATION', 'score': np.float32(0.99940175), 'word': 'Applied Data Science Politeknik Elektronika Negeri Surabaya Surabaya Indonesia Natural Science Major Sekolah Menengah Atas Surabaya Surabaya Development of a MultiNode Greenhouse Condition Dashboard Based on Data Mining as an Effort to Empower Melon Farmers in Wates District Blitar Regency Republik Melon dashtaniid December Present Backend Developer Flask Supabase cPanel Developed back', 'start': 0, 'end': 388}, {'entity_group': 'SKILL', 'score': np.float32(0.9605982), 'word': '##end', 'start': 388, 'end': 391}, {'entity_group': 'EDUCATION', 'score': np.float32(0.9914486), 'word': 'services using Flask framework to handle API requests and', 'start': 392, 'end': 449}, {'entity_group': 'EDUCATION', 'score': np.float32(0.99610835), 'word': 'Integrated Supabase for database management ensuring seamless connection and', 'start': 482, 'end': 558}, {'entity_group': 'EDUCATION', 'score': np.float

### KEYWORD MATCHING

In [21]:
def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s\-']+", " ", text.lower())).strip()

def format_skills(skills):
    """Format skill names agar rapi (huruf besar di awal)."""
    return sorted(set(s.title() for s in skills))

def refine_skills(extracted_skills, job_skills):
    """Filter hasil NER agar hanya ambil skill yang relevan dengan job requirement."""
    refined = set()
    text_all = " ".join(extracted_skills).lower()
    for skill in job_skills:
        if skill.lower() in text_all:
            refined.add(skill.title())
    return sorted(refined)

### JOB MATCHING

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def match_skills(cv_skills, job_skills, threshold=0.3):
    matched = []
    cv_texts = [s.lower() for s in cv_skills]
    job_texts = [s.lower() for s in job_skills]

    for job_skill in job_texts:
        # gabungkan skill CV + skill job untuk vectorizer
        corpus = cv_texts + [job_skill]
        vectorizer = TfidfVectorizer().fit(corpus)
        tfidf_matrix = vectorizer.transform(corpus)

        # similarity antara job_skill dengan semua skill CV
        sim_scores = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1]).flatten()

        # ambil skill CV dengan similarity tertinggi
        best_idx = sim_scores.argmax()
        best_score = sim_scores[best_idx]

        if best_score >= threshold:
            matched.append(job_skill.title())

    score = round(len(matched) / len(job_texts) * 100, 2) if job_texts else 0
    return sorted(set(matched)), score


### EXTRACTION

In [23]:
job_description = {
    "title": "Software Developer",
    "education_required": ["Bachelor's Degree in Computer Science"],
    "skills_required": ["Python", "Django", "Machine Learning", "SQL", "Spark"],
}

# Ambil hasil ekstraksi dari NER
cv_education = cv_entities.get("education", [])
cv_skills_raw = cv_entities.get("skills", [])
cv_skills = format_skills(cv_skills_raw)

# Refinement: filter skill CV agar hanya yang s dengan job requirement
cv_skills_refined = refine_skills(cv_skills_raw, job_description["skills_required"])

# Matching dengan TF-IDF + Cosine Similarity
matched_skills, match_score = match_skills(cv_skills_refined, job_description["skills_required"])

### RESULT

In [24]:
print("\n" + "="*70)
print("CV EXTRACTION RESULT")
print("="*70)
print(f"Education: {cv_education}")
print(f"Skills (refined): {cv_skills_refined}")
print(f"\nJob Match Score: {match_score}%")
print("-" * 70)
print(f"Required Skills: {job_description['skills_required']}")
print(f"Matched Skills: {matched_skills}")
print(f"Match Rate: {len(matched_skills)}/{len(job_description['skills_required'])} = {match_score}%")
print("-" * 70)


CV EXTRACTION RESULT
Education: ['Ba', 'Development with PHPImplemented the backend logic usingto process user input', 'faculty', 'Applied Data Science Politeknik Elektronika Negeri Surabaya Surabaya Indonesia Natural Science Major Sekolah Menengah Atas Surabaya Surabaya Development of a MultiNode Greenhouse Condition Dashboard Based on Data Mining as an Effort to Empower Melon Farmers in Wates District Blitar Regency Republik Melon dashtaniid December Present Backend Developer Flask Supabase cPanel Developed back', 'FullStack Developer EEPIS LEN', 'A', 'HL and', 'services using Flask framework to handle API requests andIntegrated Supabase for database management ensuring seamless connection andreDeployed machine learning models and backend applications on cPanel hosting handling', 'Database Management with phpMyAdmin MySet up and managed the My', 'handling room', 'NG SPE for Campus Room Booking System Applied Data Science of PENS February June Developed Frontend with HTML and CSSDesi